In [1]:
import pm4py
import pandas as pd

from IPython.display import Image, display
from pm4py.algo.evaluation.replay_fitness import algorithm as replay_fitness_evaluator
from pm4py.algo.evaluation.precision import algorithm as precision_evaluator
from pm4py.algo.evaluation.generalization import algorithm as generalization_evaluator
from pm4py.algo.evaluation.simplicity import algorithm as simplicity_evaluator

In [ ]:
# Load the XES file
log = pm4py.read_xes("BPI Challenge 2017.xes")

# Convert the log to a DataFrame
log_df = pm4py.convert_to_dataframe(log)

## Preprocessing

In [ ]:
# Keep only events with lifecycle=complete
if "lifecycle:transition" in log_df.columns:
    before_events = len(log_df)
    before_cases = log_df["case:concept:name"].nunique()

    log_filt_complete_events = log_df[
        log_df["lifecycle:transition"].str.lower() == "complete"
    ].copy()

    log_filt = log_filt_complete_events.copy()

    after_events = len(log_filt_complete_events)
    after_cases = log_filt_complete_events["case:concept:name"].nunique()

    print("Events before:", before_events, "| after:", after_events)
    print("Cases before:", before_cases, "| after:", after_cases)
    print("Lifecycle values:", log_df["lifecycle:transition"].dropna().value_counts().head(10))
else:
    print("No lifecycle:transition column found in this log.")

In [ ]:
# Preprocessing

case_col = "case:concept:name"
activity_col = "concept:name"
timestamp_col = "time:timestamp"

# Outcome activities used to identify completed application outcomes.
completed_end_activities = {
    "A_Cancelled",
    "A_Denied",
    "A_Pending",
}

# Use lifecycle-complete events as source if available, otherwise full log.
source_log = log_filt.copy() if "log_filt" in globals() else log_df.copy()

# Keep cases that contain at least one outcome event (not necessarily the last event).
outcome_cases = source_log[source_log[activity_col].isin(completed_end_activities)][case_col].unique()
log_filt = source_log[source_log[case_col].isin(outcome_cases)].copy()

print("Cases in source log:", source_log[case_col].nunique())
print("Cases with outcome event:", len(outcome_cases))
print("Cases after outcome-case filter:", log_filt[case_col].nunique())

Cases in source log: 31509
Cases with outcome event: 31411
Cases after outcome-case filter: 31411


## Discovery

In [ ]:
## Discovery

# Discover the Petri net process models on the filtered log
net_ind, im_ind, fm_ind = pm4py.discover_petri_net_inductive(log_filt, noise_threshold=0.2)
net_heu, im_heu, fm_heu = pm4py.discover_petri_net_heuristics(log_filt, loop_two_threshold=0.8)

# Save the discovered models
inductive_pnml = "models/model_inductive.pnml"
heuristics_pnml = "models/model_heuristics.pnml"

pm4py.write_pnml(net_ind, im_ind, fm_ind, inductive_pnml)
pm4py.write_pnml(net_heu, im_heu, fm_heu, heuristics_pnml)

# Save images
inductive_png = "results/net_inductive.png"
heuristics_png = "results/net_heuristics.png"

pm4py.save_vis_petri_net(net_ind, im_ind, fm_ind, inductive_png)
pm4py.save_vis_petri_net(net_heu, im_heu, fm_heu, heuristics_png)

print("Inductive net:")
display(Image(filename=inductive_png))

print("Heuristics net:")
display(Image(filename=heuristics_png))

## Quality Metrics & Model Evaluation


In [ ]:
# Fitness and precision on lifecycle-complete baseline

fit_ind = replay_fitness_evaluator.apply(
    log_filt_complete_events,
    net_ind,
    im_ind,
    fm_ind,
    variant=replay_fitness_evaluator.Variants.TOKEN_BASED,
)
fit_heu = replay_fitness_evaluator.apply(
    log_filt_complete_events,
    net_heu,
    im_heu,
    fm_heu,
    variant=replay_fitness_evaluator.Variants.TOKEN_BASED,
)

log_fitness_ind = fit_ind.get("log_fitness", fit_ind.get("average_trace_fitness"))
log_fitness_heu = fit_heu.get("log_fitness", fit_heu.get("average_trace_fitness"))

precision_ind = precision_evaluator.apply(
    log_filt_complete_events,
    net_ind,
    im_ind,
    fm_ind,
    variant=precision_evaluator.Variants.ETCONFORMANCE_TOKEN,
)
precision_heu = precision_evaluator.apply(
    log_filt_complete_events,
    net_heu,
    im_heu,
    fm_heu,
    variant=precision_evaluator.Variants.ETCONFORMANCE_TOKEN,
)

print("Inductive - Log fitness:", round(float(log_fitness_ind), 4))
print("Heuristics - Log fitness:", round(float(log_fitness_heu), 4))
print("Inductive - Precision:", round(float(precision_ind), 4))
print("Heuristics - Precision:", round(float(precision_heu), 4))

In [ ]:
## Quality Validation

models = {
    "inductive": (net_ind, im_ind, fm_ind),
    "heuristics": (net_heu, im_heu, fm_heu),
}

rows = []
for name, (net, im, fm) in models.items():
    
    # Simplicity and generalization
    simplicity = simplicity_evaluator.apply(net)
    generalization = generalization_evaluator.apply(log_filt_complete_events, net, im, fm)
    
    # Structural complexity
    n_places = len(net.places)
    n_trans = len(net.transitions)
    n_arcs = len(net.arcs)
    
    rows.append({
        "model": name,
        "places": n_places,
        "transitions": n_trans,
        "arcs": n_arcs,
        "simplicity": simplicity,
        "generalization": generalization,
    })

quality_df = pd.DataFrame(rows).set_index("model")
display(quality_df)

In [ ]:
## Additional Metrics

for name, (net, im, fm) in models.items():

    # Inverse Model Size
    size = len(net.places) + len(net.transitions) + len(net.arcs)
    quality_df.loc[name, "inverse_model_size"] = 1.0 / (1.0 + size)
    
    # Inverse Average Branching
    avg_branch = len(net.arcs) / (len(net.places) + len(net.transitions))
    quality_df.loc[name, "inverse_average_branching"] = 1.0 / (1.0 + avg_branch)

display(quality_df)